# Data Preprocessing

In [1]:
# import dependencies
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [2]:
# import data and merge
data_dir = '../data/'
sold_files = [
    'CRMLSSold202512.csv',
    'CRMLSSold202601.csv',
    'CRMLSSold202602.csv',
    'CRMLSSold202603.csv',
    'CRMLSSold202604.csv',
    'CRMLSSold202605.csv',
    'CRMLSSold202606.csv'
]

dfs = [pd.read_csv(os.path.join(data_dir, file_name)) for file_name in sold_files]

# merge together
data = pd.concat(dfs, ignore_index=True)

# convert close date and create close month
data['CloseDate'] = pd.to_datetime(data['CloseDate'], errors='coerce')
data['CloseMonth'] = data['CloseDate'].dt.to_period('M')

/var/folders/zr/3bj1t2vd2klbq4dnl21sqz7m0000gn/T/ipykernel_69562/1300538776.py:13: DtypeWarning: Columns (0: WaterfrontYN, 1: PostalCode) have mixed types. Specify dtype option on import or set low_memory=False.
  dfs = [pd.read_csv(os.path.join(data_dir, file_name)) for file_name in sold_files]


In [3]:
# restrict analysis to California residential single-family properties
state_col = 'StateOrProvince'
if state_col not in data.columns:
    raise KeyError(f'{state_col} is required to restrict the data to CA properties')

ca_mask = data[state_col].astype('string').str.strip().str.upper().eq('CA')
data_ca = data[ca_mask].copy()

data_res = data_ca[
    (data_ca['PropertyType'] == 'Residential') &
    (data_ca['PropertySubType'] == 'SingleFamilyResidence')
].copy()

print('Original shape:', data.shape)
print('CA shape:', data_ca.shape)
print('CA residential single-family shape:', data_res.shape)

data_res.head()

Original shape: (148911, 79)
CA shape: (148902, 79)
CA residential single-family shape: (74583, 79)


,BuyerAgentAOR,ListAgentAOR,Flooring,ViewYN,WaterfrontYN,BasementYN,PoolPrivateYN,OriginalListPrice,ListingKey,ListAgentEmail,...,LotSizeArea,MainLevelBedrooms,NewConstructionYN,GarageSpaces,HighSchoolDistrict,PostalCode,AssociationFee,LotSizeSquareFeet,MiddleOrJuniorSchoolDistrict,CloseMonth
0,ContraCosta,ContraCosta,"Carpet,Tile,Wood",NaN,NaN,NaN,False,1998000.0,1150041639,teresa@teresahooper.com,...,10080.0,NaN,False,3.0,San Ramon Valley,94596,975.0,10080.0,NaN,2025-12
2,SanDiego,SanDiego,"Carpet,Wood",True,NaN,NaN,False,2214421.0,1150038683,laura@lauralothianrealestate.com,...,34745.0,NaN,False,3.0,NaN,91364,NaN,34745.0,NaN,2025-12
3,Mlslistings,Mlslistings,NaN,False,NaN,NaN,NaN,1200000.0,1150038607,trung.lam@kw.com,...,6600.0,NaN,False,2.0,Other,95121,NaN,6600.0,NaN,2025-12
7,Mlslistings,Mlslistings,NaN,False,NaN,NaN,NaN,3100000.0,1150032869,vickie@realsmartgroup.com,...,8262.0,NaN,False,1.0,San Jose Unified,95124,NaN,8262.0,NaN,2025-12
9,Mlslistings,Mlslistings,NaN,False,NaN,NaN,NaN,2900000.0,1150028403,vickie@realsmartgroup.com,...,9222.0,NaN,False,2.0,Other,95128,NaN,9222.0,NaN,2025-12


## Select Features

In [4]:
# target, date, numeric, categorical, and boolean/categorical fields
target_col = 'ClosePrice'
date_cols = ['CloseDate', 'CloseMonth']

numeric_features = [
    'LivingArea',
    'BedroomsTotal',
    'BathroomsTotalInteger',
    'LotSizeAcres',
    'YearBuilt',
    'GarageSpaces',
    'ParkingTotal',
    'TaxAnnualAmount',
    'AssociationFee'
]

categorical_features = [
    'City',
    'CountyOrParish',
    'PostalCode',
    'MLSAreaMajor',
    'HighSchoolDistrict',
    'ElementarySchoolDistrict',
    'MiddleOrJuniorSchoolDistrict',
    'Stories',
    'Levels'
]

binary_categorical_features = [
    'FireplaceYN',
    'BasementYN',
    'PoolPrivateYN',
    'AttachedGarageYN',
    'NewConstructionYN'
]

# retained for feature engineering in later notebooks, not current model encoding
feature_engineering_source_cols = [
    'StateOrProvince',
    'Latitude',
    'Longitude',
    'LotSizeSquareFeet',
    'LotSizeArea',
    'MainLevelBedrooms',
    'ElementarySchool',
    'MiddleOrJuniorSchool',
    'HighSchool',
    'ViewYN',
    'Flooring'
]

# these fields are excluded because they can cause data leakage
leakage_columns = [
    'ListPrice',
    'OriginalListPrice',
    'DaysOnMarket',
    'PurchaseContractDate'
]

In [5]:
# keep only a subset of the columns
existing_date_cols = [col for col in date_cols if col in data_res.columns]
existing_numeric_features = [col for col in numeric_features if col in data_res.columns]
existing_categorical_features = [col for col in categorical_features + binary_categorical_features if col in data_res.columns]
existing_feature_engineering_source_cols = [col for col in feature_engineering_source_cols if col in data_res.columns]
req_columns = list(dict.fromkeys(
    existing_date_cols +
    existing_numeric_features +
    existing_categorical_features +
    existing_feature_engineering_source_cols +
    [target_col]
))
subset = [col for col in req_columns if col in data_res.columns and col not in leakage_columns]

model_data = data_res[subset].copy()

excluded_present = [col for col in leakage_columns if col in data_res.columns]

print('Selected columns:', len(subset))
print()
print(subset)
print()
print('Feature engineering source columns retained:', existing_feature_engineering_source_cols)
print()
print('Leakage columns found and excluded:', excluded_present)

Selected columns: 37

['CloseDate', 'CloseMonth', 'LivingArea', 'BedroomsTotal', 'BathroomsTotalInteger', 'LotSizeAcres', 'YearBuilt', 'GarageSpaces', 'ParkingTotal', 'TaxAnnualAmount', 'AssociationFee', 'City', 'CountyOrParish', 'PostalCode', 'MLSAreaMajor', 'HighSchoolDistrict', 'ElementarySchoolDistrict', 'MiddleOrJuniorSchoolDistrict', 'Stories', 'Levels', 'FireplaceYN', 'BasementYN', 'PoolPrivateYN', 'AttachedGarageYN', 'NewConstructionYN', 'StateOrProvince', 'Latitude', 'Longitude', 'LotSizeSquareFeet', 'LotSizeArea', 'MainLevelBedrooms', 'ElementarySchool', 'MiddleOrJuniorSchool', 'HighSchool', 'ViewYN', 'Flooring', 'ClosePrice']

Feature engineering source columns retained: ['StateOrProvince', 'Latitude', 'Longitude', 'LotSizeSquareFeet', 'LotSizeArea', 'MainLevelBedrooms', 'ElementarySchool', 'MiddleOrJuniorSchool', 'HighSchool', 'ViewYN', 'Flooring']

Leakage columns found and excluded: ['ListPrice', 'OriginalListPrice', 'DaysOnMarket', 'PurchaseContractDate']


## Initial Cleaning

In [6]:
# drop WaterfrontYN because it is almost entirely missing
if 'WaterfrontYN' in model_data.columns:
    model_data = model_data.drop(columns=['WaterfrontYN'])
    print('Dropped WaterfrontYN')
else:
    print('WaterfrontYN was not in the retained data')

WaterfrontYN was not in the retained data


In [7]:
# convert numeric columns to numeric dtype
numeric_cleaning_cols = [col for col in existing_numeric_features + [target_col] if col in model_data.columns]

for col in numeric_cleaning_cols:
    model_data[col] = pd.to_numeric(model_data[col], errors='coerce')

In [8]:
# clean PostalCode by keeping only the first 5 digits
if 'PostalCode' in model_data.columns:
    zip_5 = model_data['PostalCode'].astype(str).str.extract(r'(\d{5})', expand=False)
    model_data['PostalCode'] = zip_5.where(zip_5.str.len() == 5, np.nan)

model_data[['PostalCode']].head() if 'PostalCode' in model_data.columns else 'PostalCode not found'

,PostalCode
0,94596
2,91364
3,95121
7,95124
9,95128


In [9]:
# replace unrealistic values with missing values
impossible_value_rules = {
    'LivingArea': 0,
    'ClosePrice': 10000,  # too low for close price
    'LotSizeAcres': 0,
    'BedroomsTotal': 0,
    'BathroomsTotalInteger': 0
}

for col, min_value in impossible_value_rules.items():
    if col in model_data.columns:
        model_data.loc[model_data[col] <= min_value, col] = np.nan

before_drop = model_data.shape[0]
model_data = model_data.dropna(subset=['ClosePrice', 'CloseDate', 'CloseMonth']).copy()
print('Rows dropped because target/date was missing after cleaning:', before_drop - model_data.shape[0])
print('Shape after initial cleaning:', model_data.shape)

Rows dropped because target/date was missing after cleaning: 5
Shape after initial cleaning: (74578, 37)


The EDA notebook showed skewed data and outliers. I will cap selected outliers using quantiles.

In [10]:
# columns that will be capped later using final training data only
cap_99_columns = [
    'LivingArea',
    'LotSizeAcres',
    'BedroomsTotal',
    'BathroomsTotalInteger',
    'TaxAnnualAmount',
    'AssociationFee'
]
cap_99_columns = [col for col in cap_99_columns if col in model_data.columns]

print('Feature columns to cap at the training 99th percentile:')
print(cap_99_columns)
print('\nClosePrice will be capped at the training 1st and 99th percentiles.')

Feature columns to cap at the training 99th percentile:
['LivingArea', 'LotSizeAcres', 'BedroomsTotal', 'BathroomsTotalInteger', 'TaxAnnualAmount', 'AssociationFee']

ClosePrice will be capped at the training 1st and 99th percentiles.


## Handling Missing Values

In [11]:
# show missingness table for selected features
selected_feature_cols = [
    col for col in existing_numeric_features + existing_categorical_features
    if col in model_data.columns and col not in leakage_columns
]

missing_table = pd.DataFrame({
    'missing_count': model_data[selected_feature_cols].isna().sum(),
    'missing_percent': model_data[selected_feature_cols].isna().mean() * 100
}).sort_values('missing_percent', ascending=False)

missing_table

,missing_count,missing_percent
MiddleOrJuniorSchoolDistrict,74578,100.000000
TaxAnnualAmount,74578,100.000000
ElementarySchoolDistrict,74578,100.000000
BasementYN,72801,97.617260
AssociationFee,21477,28.798037
HighSchoolDistrict,20210,27.099145
MLSAreaMajor,10709,14.359463
AttachedGarageYN,9059,12.147014
Stories,7906,10.600982
PoolPrivateYN,5893,7.901794


For the missing values:

- Numeric columns will be imputed using the training median.
- Categorical columns will be filled with `"Unknown"`.
- Missing indicator columns are created to preserve whether a numeric value was originally missing.

I do not impute with the full dataset before the split because that would leak information from the test month into training.

In [12]:
# create missing indicator columns for numeric features with meaningful missingness
numeric_missing_rates = model_data[existing_numeric_features].isna().mean()
numeric_missing_cols = numeric_missing_rates[
    (numeric_missing_rates > 0.01) & (numeric_missing_rates < 1.00)
].index.tolist()

for col in numeric_missing_cols:
    model_data[col + '_missing'] = model_data[col].isna().astype(int)

missing_indicator_cols = [col + '_missing' for col in numeric_missing_cols]

print('Numeric columns with missing indicators:')
print(missing_indicator_cols)

Numeric columns with missing indicators:
['LotSizeAcres_missing', 'GarageSpaces_missing', 'AssociationFee_missing']


## Train/Test Split

The most recent available month, June 2026, is used as the test set. For training, I test different values of `X`, where `X` is the number of months immediately before the test month.

In [13]:
# identify available months and use the most recent month as the test month
available_months = sorted(model_data['CloseMonth'].dropna().unique())
test_month = available_months[-1]
candidate_x_values = list(range(2, len(available_months)))

# inspect each candidate training window
candidate_splits = {}
key_missing_cols = [
    col for col in ['ClosePrice', 'LivingArea', 'BedroomsTotal', 'BathroomsTotalInteger', 'LotSizeAcres']
    if col in model_data.columns
]

for x in candidate_x_values:
    train_months = available_months[-(x + 1):-1]
    train_candidate = model_data[model_data['CloseMonth'].isin(train_months)].copy()
    test_candidate = model_data[model_data['CloseMonth'] == test_month].copy()
    
    candidate_splits[x] = {
        'train_months': train_months,
        'train': train_candidate,
        'test': test_candidate
    }
    
    print('\n' + '=' * 60)
    print('X =', x)
    print('Training months:', train_months)
    print('Test month:', test_month)
    print('Train date range:', train_candidate['CloseDate'].min(), 'to', train_candidate['CloseDate'].max())
    print('Test date range:', test_candidate['CloseDate'].min(), 'to', test_candidate['CloseDate'].max())
    print('Train shape:', train_candidate.shape)
    print('Test shape:', test_candidate.shape)
    
    price_summary = pd.DataFrame({
        'train': train_candidate['ClosePrice'].agg(['mean', 'median', 'std']),
        'test': test_candidate['ClosePrice'].agg(['mean', 'median', 'std'])
    })
    print('\nClosePrice summary:')
    print(price_summary)
    
    missing_summary = pd.DataFrame({
        'train_missing_percent': train_candidate[key_missing_cols].isna().mean() * 100,
        'test_missing_percent': test_candidate[key_missing_cols].isna().mean() * 100
    })
    print('\nMissingness for key columns:')
    print(missing_summary)


X = 2
Training months: [Period('2026-04', 'M'), Period('2026-05', 'M')]
Test month: 2026-06
Train date range: 2026-04-01 00:00:00 to 2026-05-31 00:00:00
Test date range: 2026-06-01 00:00:00 to 2026-06-30 00:00:00
Train shape: (24053, 40)
Test shape: (12856, 40)

ClosePrice summary:
               train          test
mean    1.406268e+06  1.306106e+06
median  9.250000e+05  9.250000e+05
std     8.402521e+06  1.536523e+06

Missingness for key columns:
                       train_missing_percent  test_missing_percent
ClosePrice                          0.000000              0.000000
LivingArea                          0.078992              0.101120
BedroomsTotal                       0.041575              0.062228
BathroomsTotalInteger               0.033260              0.062228
LotSizeAcres                        2.103688              1.866833

X = 3
Training months: [Period('2026-03', 'M'), Period('2026-04', 'M'), Period('2026-05', 'M')]
Test month: 2026-06
Train date range: 2026-03-0

I choose the training window using data quality checks. I compare the number of rows, missingness, and how similar the train and test `ClosePrice` medians are.

In [14]:
# summarize candidate windows
summary_rows = []

for x, split_info in candidate_splits.items():
    train_candidate = split_info['train']
    test_candidate = split_info['test']
    train_months = split_info['train_months']
    
    summary_rows.append({
        'X': x,
        'training_months_used': ', '.join(str(month) for month in train_months),
        'train_rows': train_candidate.shape[0],
        'test_rows': test_candidate.shape[0],
        'train_median_ClosePrice': train_candidate['ClosePrice'].median(),
        'test_median_ClosePrice': test_candidate['ClosePrice'].median(),
        'abs_diff_train_test_median_ClosePrice': abs(
            train_candidate['ClosePrice'].median() - test_candidate['ClosePrice'].median()
        ),
        'avg_missingness_selected_features': train_candidate[selected_feature_cols].isna().mean().mean() * 100
    })

window_summary = pd.DataFrame(summary_rows)
window_summary

,X,training_months_used,train_rows,test_rows,train_median_ClosePrice,test_median_ClosePrice,abs_diff_train_test_median_ClosePrice,avg_missingness_selected_features
0,2,"2026-04, 2026-05",24053,12856,925000.0,925000.0,0.0,22.792420
1,3,"2026-03, 2026-04, 2026-05",35230,12856,915000.0,925000.0,10000.0,22.713473
2,4,"2026-02, 2026-03, 2026-04, 2026-05",43779,12856,905000.0,925000.0,20000.0,22.673964
3,5,"2026-01, 2026-02, 2026-03, 2026-04, 2026-05",51267,12856,900000.0,925000.0,25000.0,22.616464
4,6,"2025-12, 2026-01, 2026-02, 2026-03, 2026-04, 2...",61722,12856,890000.0,925000.0,35000.0,22.569079


I will use `X = 5` because it keeps a five-month rolling training window immediately before the June 2026 test set. This shifts the final training window to January through May 2026 while leaving December 2025 available for comparison as `X = 6`.

In [15]:
final_x = 5
final_train_months = candidate_splits[final_x]['train_months']

train_raw = candidate_splits[final_x]['train'].copy()
test_raw = candidate_splits[final_x]['test'].copy()

print('Final X:', final_x)
print('Final train shape:', train_raw.shape)
print('Final test shape:', test_raw.shape)
print(f'Percent split: train - {(train_raw.shape[0]/(train_raw.shape[0]+test_raw.shape[0]))*100:.2f}%, test - {(test_raw.shape[0]/(train_raw.shape[0]+test_raw.shape[0]))*100:.2f}%')

Final X: 5
Final train shape: (51267, 40)
Final test shape: (12856, 40)
Percent split: train - 79.95%, test - 20.05%


## Preprocessing

In [16]:
# cap outliers using training data only
train_clean = train_raw.copy()
test_clean = test_raw.copy()
cap_values = []

for col in cap_99_columns:
    upper = train_clean[col].quantile(0.99)
    if pd.notna(upper):
        train_clean[col] = train_clean[col].clip(upper=upper)
        test_clean[col] = test_clean[col].clip(upper=upper)
        cap_values.append({'column': col, 'lower_cap': np.nan, 'upper_cap': upper})

# cap target using training data only
close_lower = train_clean[target_col].quantile(0.01)
close_upper = train_clean[target_col].quantile(0.99)
train_clean[target_col] = train_clean[target_col].clip(lower=close_lower, upper=close_upper)
test_clean[target_col] = test_clean[target_col].clip(lower=close_lower, upper=close_upper)
cap_values.append({'column': target_col, 'lower_cap': close_lower, 'upper_cap': close_upper})

cap_values = pd.DataFrame(cap_values)
cap_values

,column,lower_cap,upper_cap
0,LivingArea,NaN,5.713750e+03
1,LotSizeAcres,NaN,6.796100e+00
2,BedroomsTotal,NaN,6.000000e+00
3,BathroomsTotalInteger,NaN,6.000000e+00
4,AssociationFee,NaN,1.160000e+03
5,ClosePrice,235000.0,6.500000e+06


I also convert categorical columns to strings before encoding. Missing values are still left as missing here because the pipeline will fill them with `"Unknown"`.

In [17]:
# final feature lists
numeric_model_features = [col for col in existing_numeric_features + missing_indicator_cols if col in train_clean.columns]
categorical_model_features = [col for col in existing_categorical_features if col in train_clean.columns]

# drop columns that are completely missing in the training set
all_missing_numeric = [col for col in numeric_model_features if train_clean[col].isna().all()]
all_missing_categorical = [col for col in categorical_model_features if train_clean[col].isna().all()]

numeric_model_features = [col for col in numeric_model_features if col not in all_missing_numeric]
categorical_model_features = [col for col in categorical_model_features if col not in all_missing_categorical]

# make categorical fields consistent for the encoder
for col in categorical_model_features:
    train_clean[col] = train_clean[col].apply(lambda x: str(x).strip() if pd.notna(x) else np.nan)
    test_clean[col] = test_clean[col].apply(lambda x: str(x).strip() if pd.notna(x) else np.nan)
    train_clean.loc[train_clean[col] == '', col] = np.nan
    test_clean.loc[test_clean[col] == '', col] = np.nan

print('Dropped all-missing numeric features:', all_missing_numeric)
print('Dropped all-missing categorical features:', all_missing_categorical)
print()
print('Numeric model features:', numeric_model_features)
print()
print('Categorical model features:', categorical_model_features)

Dropped all-missing numeric features: ['TaxAnnualAmount']
Dropped all-missing categorical features: ['ElementarySchoolDistrict', 'MiddleOrJuniorSchoolDistrict']

Numeric model features: ['LivingArea', 'BedroomsTotal', 'BathroomsTotalInteger', 'LotSizeAcres', 'YearBuilt', 'GarageSpaces', 'ParkingTotal', 'AssociationFee', 'LotSizeAcres_missing', 'GarageSpaces_missing', 'AssociationFee_missing']

Categorical model features: ['City', 'CountyOrParish', 'PostalCode', 'MLSAreaMajor', 'HighSchoolDistrict', 'Stories', 'Levels', 'FireplaceYN', 'BasementYN', 'PoolPrivateYN', 'AttachedGarageYN', 'NewConstructionYN']


In [18]:
# group rare categories using the training set only
min_category_count = 50
high_cardinality_cutoff = 50
category_levels = {}

for col in categorical_model_features:
    n_unique_train = train_clean[col].nunique(dropna=True)
    if n_unique_train > high_cardinality_cutoff:
        common_levels = train_clean[col].value_counts().loc[lambda s: s >= min_category_count].index
        category_levels[col] = set(common_levels)
        train_clean[col] = train_clean[col].where(train_clean[col].isin(category_levels[col]), 'Other')
        test_clean[col] = test_clean[col].where(test_clean[col].isin(category_levels[col]), 'Other')

print('High-cardinality columns grouped:')
print(list(category_levels.keys()))

High-cardinality columns grouped:
['City', 'CountyOrParish', 'PostalCode', 'MLSAreaMajor', 'HighSchoolDistrict']


In [19]:
# separate features from target and reference date columns
feature_cols = numeric_model_features + categorical_model_features
reference_cols = [col for col in ['CloseDate', 'CloseMonth'] if col in train_clean.columns]

X_train = train_clean[feature_cols].copy()
X_test = test_clean[feature_cols].copy()
y_train = train_clean[target_col].copy()
y_test = test_clean[target_col].copy()

# numeric and categorical preprocessing
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median', keep_empty_features=True)),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_model_features),
    ('cat', categorical_transformer, categorical_model_features)
])

# fit on training data only, then transform train and test
X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)

print('Encoded train shape:', X_train_encoded.shape)
print('Encoded test shape:', X_test_encoded.shape)

Encoded train shape: (51267, 1087)
Encoded test shape: (12856, 1087)


In [20]:
# get transformed feature names
encoded_feature_names = preprocessor.get_feature_names_out()
encoded_feature_names = [name.replace('num__', '').replace('cat__', '') for name in encoded_feature_names]

train_encoded = pd.DataFrame(X_train_encoded, columns=encoded_feature_names, index=train_clean.index)
test_encoded = pd.DataFrame(X_test_encoded, columns=encoded_feature_names, index=test_clean.index)

# add target and date reference columns back to the encoded files
for col in reference_cols:
    train_encoded[col] = train_clean[col].astype(str).values
    test_encoded[col] = test_clean[col].astype(str).values

train_encoded[target_col] = y_train.values
test_encoded[target_col] = y_test.values

train_encoded.head()

,LivingArea,BedroomsTotal,BathroomsTotalInteger,LotSizeAcres,YearBuilt,GarageSpaces,ParkingTotal,AssociationFee,LotSizeAcres_missing,GarageSpaces_missing,...,PoolPrivateYN_Unknown,AttachedGarageYN_False,AttachedGarageYN_True,AttachedGarageYN_Unknown,NewConstructionYN_False,NewConstructionYN_True,NewConstructionYN_Unknown,CloseDate,CloseMonth,ClosePrice
20538,2.797692,0.541236,3.199883,0.910182,1.611393,-0.003622,-0.050435,-0.419918,-0.146903,-0.198635,...,1.0,1.0,0.0,0.0,1.0,0.0,0.0,2026-01-02,2026-01,6500000.0
20540,-0.477581,-1.601762,-1.552361,5.807078,0.128318,-0.565823,-0.150111,-0.419918,-0.146903,-0.198635,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,2026-01-31,2026-01,2975000.0
20542,-1.192834,-1.601762,-1.552361,-0.215869,-1.065376,-0.003622,-0.050435,-0.419918,-0.146903,-0.198635,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,2026-01-30,2026-01,720000.0
20543,-0.836826,-0.530263,-1.552361,-0.255344,-0.812169,-0.003622,-0.050435,-0.419918,-0.146903,-0.198635,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,2026-01-29,2026-01,600000.0
20546,-0.132361,0.541236,1.298986,-0.335091,1.756084,-0.003622,-0.050435,0.835615,-0.146903,-0.198635,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,2026-01-30,2026-01,6500000.0


## Save Cleaned CSV Files

I save two versions of the cleaned train and test data. The first version is before one-hot encoding. The second version is encoded and scaled for modeling.

In [21]:
# columns to save in the non-encoded cleaned files
feature_engineering_save_cols = [
    col for col in existing_feature_engineering_source_cols
    if col in train_clean.columns and col not in feature_cols and col not in reference_cols
]
not_encoded_cols = list(dict.fromkeys(reference_cols + feature_cols + feature_engineering_save_cols + [target_col]))
not_encoded_cols = [col for col in not_encoded_cols if col not in leakage_columns]

train_not_encoded = train_clean[not_encoded_cols].copy()
test_not_encoded = test_clean[not_encoded_cols].copy()

# impute numeric columns with training medians for readable files
train_numeric_medians = train_clean[numeric_model_features].median()

for col in numeric_model_features:
    train_not_encoded[col] = train_not_encoded[col].fillna(train_numeric_medians[col])
    test_not_encoded[col] = test_not_encoded[col].fillna(train_numeric_medians[col])

# fill categorical columns with Unknown for readable files
for col in categorical_model_features:
    train_not_encoded[col] = train_not_encoded[col].fillna('Unknown')
    test_not_encoded[col] = test_not_encoded[col].fillna('Unknown')

# make CloseMonth easier to read in CSV files
if 'CloseMonth' in train_not_encoded.columns:
    train_not_encoded['CloseMonth'] = train_not_encoded['CloseMonth'].astype(str)
    test_not_encoded['CloseMonth'] = test_not_encoded['CloseMonth'].astype(str)

# save non-encoded cleaned files
cleaned_data_dir = './data_clean/'
os.makedirs(cleaned_data_dir, exist_ok=True)

train_not_encoded.to_csv(os.path.join(cleaned_data_dir, 'cleaned_train_not_encoded.csv'), index=False)
test_not_encoded.to_csv(os.path.join(cleaned_data_dir, 'cleaned_test_not_encoded.csv'), index=False)

# save encoded files
train_encoded.to_csv(os.path.join(cleaned_data_dir, 'cleaned_train_encoded.csv'), index=False)
test_encoded.to_csv(os.path.join(cleaned_data_dir, 'cleaned_test_encoded.csv'), index=False)

print('Saved files:')
print(os.path.join(cleaned_data_dir, 'cleaned_train_not_encoded.csv'))
print(os.path.join(cleaned_data_dir, 'cleaned_test_not_encoded.csv'))
print(os.path.join(cleaned_data_dir, 'cleaned_train_encoded.csv'))
print(os.path.join(cleaned_data_dir, 'cleaned_test_encoded.csv'))

Saved files:
./data_clean/cleaned_train_not_encoded.csv
./data_clean/cleaned_test_not_encoded.csv
./data_clean/cleaned_train_encoded.csv
./data_clean/cleaned_test_encoded.csv
